# 02 · Classifier: EfficientNet

**Goal:** train a 4-class brain tumor classifier (glioma, meningioma, notumor, pituitary) on the clean splits from `01_data_prep.ipynb`.

**Target:** accuracy and macro-F1 **≥ 95%** on the test split.

**How we stay honest:** we tune on **val only**. The test split stays closed (`RUN_TEST = False`) until the final model is picked, then it is run once.

**Presets** (pick one in the config cell):

| Preset | Backbone | Image | Extras |
|---|---|---|---|
| `baseline` | EfficientNet-B0 | 224 px | light augmentation |
| `strong` | EfficientNet-B3 | 384 px | stronger augmentation, label smoothing, warmup |

**Run on Kaggle:**
1. *Settings → Accelerator:* GPU (T4 or P100). *Settings → Internet:* on (downloads the pretrained weights).
2. *Add Input:* `masoudnickparvar/brain-tumor-mri-dataset`.
3. *Add Input:* a dataset with `splits.csv` (the output of notebook 01).
4. Run all. Outputs land in `/kaggle/working`.

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset

SMOKE = os.environ.get("NEUROLENS_SMOKE") == "1"  # tiny local test run on fake data


def find_data_dir() -> Path:
    if "NEUROLENS_DATA" in os.environ:
        return Path(os.environ["NEUROLENS_DATA"])
    # Kaggle mounts each dataset under /kaggle/input; the folder holding Training/ is the root
    for p in Path("/kaggle/input").glob("**/Training"):
        return p.parent
    raise FileNotFoundError("Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset.")


def find_splits() -> Path:
    if "NEUROLENS_SPLITS" in os.environ:
        return Path(os.environ["NEUROLENS_SPLITS"])
    for p in [*Path("/kaggle/input").glob("**/splits.csv"), Path("../data/splits.csv")]:
        if p.exists():
            return p
    raise FileNotFoundError("splits.csv not found. Attach the output of 01_data_prep.ipynb.")


DATA_DIR, SPLITS_CSV = find_data_dir(), find_splits()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| splits:", SPLITS_CSV)

## 1. Config

In [ ]:
PRESET = os.environ.get("NEUROLENS_PRESET", "strong")  # "baseline" or "strong"

PRESETS = {
    "baseline": dict(model="efficientnet_b0", img_size=224, batch_size=32, lr=3e-4,
                     label_smoothing=0.0, strong_aug=False, warmup_epochs=0, patience=6),
    "strong":   dict(model="efficientnet_b3", img_size=384, batch_size=24, lr=3e-4,
                     label_smoothing=0.1, strong_aug=True, warmup_epochs=1, patience=7),
}
CFG = dict(preset=PRESET, epochs=30, weight_decay=1e-4, num_workers=4, seed=42, pretrained=True, **PRESETS[PRESET])
if SMOKE:
    CFG.update(img_size=64, batch_size=8, epochs=2, num_workers=0, pretrained=False)

RUN_TEST = False  # open the test split only once, for the final model
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"  # mixed precision: faster and lighter on GPU
print("device:", device, "| amp:", use_amp)
print(CFG)

## 2. Data

In [ ]:
df = pd.read_csv(SPLITS_CSV)
df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})
train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
df.groupby(["split", "label"]).size().unstack(fill_value=0)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet stats, to match the pretrained backbone
S = CFG["img_size"]

if CFG["strong_aug"]:
    train_aug = [
        A.HorizontalFlip(p=0.5),  # brains are roughly left-right symmetric
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-20, 20), p=0.8),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
        A.RandomGamma(p=0.3),        # the dataset has the same scans with other contrast curves
        A.GaussNoise(p=0.2),
        A.CoarseDropout(p=0.3),      # small cut-out holes: don't rely on one spot
    ]
else:
    train_aug = [
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.7),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    ]

train_tf = A.Compose([A.Resize(S, S), *train_aug, A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
eval_tf = A.Compose([A.Resize(S, S), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])


class MRIDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tf: A.Compose):
        self.paths = [DATA_DIR / p for p in frame.path]
        self.ys = frame.y.tolist()
        self.tf = tf

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert("RGB"))  # grayscale MRI -> 3 channels for the pretrained net
        return self.tf(image=img)["image"], self.ys[i]


def make_loader(frame, tf, shuffle):
    return DataLoader(MRIDataset(frame, tf), batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=use_amp, drop_last=shuffle)


train_loader = make_loader(train_df, train_tf, shuffle=True)
val_loader = make_loader(val_df, eval_tf, shuffle=False)

### Check a few augmented training images

In [ ]:
x, y = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img, label in zip(axes, x[:6], y[:6]):
    img = img.permute(1, 2, 0).numpy() * STD + MEAN  # undo normalize for display
    ax.imshow(img.clip(0, 1))
    ax.set_title(CLASSES[label])
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "augmented_examples.png", dpi=80)
plt.show()

## 3. Model

In [ ]:
model = timm.create_model(CFG["model"], pretrained=CFG["pretrained"], num_classes=len(CLASSES)).to(device)

# Class weights so a bigger class doesn't dominate the loss; label smoothing keeps the model from being over-confident
counts = train_df.y.value_counts().sort_index().to_numpy()
class_weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CFG["label_smoothing"])

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])


def lr_factor(epoch: int) -> float:
    """Linear warmup for the first epochs, then cosine decay to zero."""
    w, total = CFG["warmup_epochs"], CFG["epochs"]
    if epoch < w:
        return (epoch + 1) / (w + 1)
    return 0.5 * (1 + np.cos(np.pi * (epoch - w) / max(1, total - w)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
print(f"{CFG['model']}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

In [ ]:
def run_epoch(loader, train: bool):
    """One pass over the loader. Returns loss, accuracy, macro-F1, labels, predictions, probabilities."""
    model.train(train)
    total_loss, ys, probs = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)  # loss in float32: fp16 logits can overflow to inf/NaN
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * len(y)
            ys += y.tolist()
            probs.append(logits.float().softmax(1).detach().cpu())
    probs = torch.cat(probs).numpy()
    preds = probs.argmax(1)
    return total_loss / len(ys), accuracy_score(ys, preds), f1_score(ys, preds, average="macro"), np.array(ys), preds, probs

## 4. Train (early stopping on val macro-F1)

In [ ]:
CKPT = OUT_DIR / "classifier_best.pt"
best_f1, best_epoch, bad_epochs, history = -1.0, 0, 0, []

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_f1, *_ = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1, *_ = run_epoch(val_loader, train=False)
    scheduler.step()
    history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss, val_acc=va_acc, val_f1=va_f1))
    print(f"epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} f1 {va_f1:.3f} | {time.time() - t0:.0f}s")

    if va_f1 > best_f1:
        best_f1, best_epoch, bad_epochs = va_f1, epoch, 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad_epochs += 1
        if bad_epochs >= CFG["patience"]:
            print(f"early stop: no val F1 gain for {CFG['patience']} epochs")
            break

hist = pd.DataFrame(history)
hist.to_csv(OUT_DIR / "history.csv", index=False)
print(f"best val macro-F1 {best_f1:.4f} at epoch {best_epoch}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist.epoch, hist.train_loss, label="train")
ax1.plot(hist.epoch, hist.val_loss, label="val")
ax1.set_title("loss")
ax1.legend()
ax2.plot(hist.epoch, hist.train_acc, label="train acc")
ax2.plot(hist.epoch, hist.val_acc, label="val acc")
ax2.plot(hist.epoch, hist.val_f1, label="val macro-F1")
ax2.set_title("metrics")
ax2.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "curves.png", dpi=80)
plt.show()

## 5. Evaluate the best checkpoint on val

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))


def report(name: str, frame: pd.DataFrame) -> dict:
    loss, acc, f1, ys, preds, probs = run_epoch(make_loader(frame, eval_tf, shuffle=False), train=False)
    print(f"{name}: acc {acc:.4f} | macro-F1 {f1:.4f}")
    print(classification_report(ys, preds, labels=range(len(CLASSES)), target_names=CLASSES, digits=4, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(ys, preds, labels=range(len(CLASSES)), display_labels=CLASSES, cmap="Blues", colorbar=False)
    plt.title(f"{name} confusion matrix")
    plt.savefig(OUT_DIR / f"confusion_{name}.png", dpi=80, bbox_inches="tight")
    plt.show()

    # Wrong predictions, most confident first: these show what the model still gets wrong
    wrong = np.nonzero(preds != ys)[0]
    wrong = wrong[np.argsort(-probs[wrong].max(1))][:12]
    if len(wrong):
        fig, axes = plt.subplots(2, 6, figsize=(18, 6), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, i in zip(axes.flat, wrong):
            ax.imshow(Image.open(DATA_DIR / frame.path[i]).convert("L"), cmap="gray")
            ax.set_title(f"true {CLASSES[ys[i]]}\npred {CLASSES[preds[i]]} ({probs[i].max():.0%})", fontsize=9)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"errors_{name}.png", dpi=70)
        plt.show()
    return dict(acc=round(acc, 4), macro_f1=round(f1, 4), errors=int((preds != ys).sum()))


val_metrics = report("val", val_df)

## 6. Test (only for the final model)

In [ ]:
test_metrics = None
if RUN_TEST:
    test_metrics = report("test", test_df)
else:
    print("Test split closed. Set RUN_TEST = True only for the final model.")

## 7. Save

In [ ]:
metrics = dict(config=CFG, best_epoch=best_epoch, val=val_metrics, test=test_metrics,
               n_train=len(train_df), n_val=len(val_df), n_test=len(test_df))
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

**Next:** download `metrics.json`, `history.csv`, `curves.png`, `confusion_val.png` and `errors_val.png` from the Output tab. The weights (`classifier_best.pt`) stay on Kaggle, they are too big for git.